<a href="https://colab.research.google.com/github/pujaroy280/DATA612/blob/main/DATA_612_Project_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Project 4: Accurary & Beyond**
Puja Roy

Summer 2026


## **Introduction**

The goal of this project was to practice working with accuracy and other recommendentation system metrics by completing deliverables and applying another dataset other than the MovieLens dataset. For this project, the Anime Recommendations Database Kaggle dataset was utilized:
https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database

## **PART 1: SETUP + DATA**

In [1]:
# Install surprise library
!pip install surprise

# Import libraries for data handling and recommender systems
import pandas as pd                      # data manipulation
import numpy as np                       # numerical operations

from surprise import Dataset             # Surprise dataset handler
from surprise import Reader              # defines rating scale
from surprise import SVD                 # matrix factorization model
from surprise import KNNBasic            # collaborative filtering model
from surprise.model_selection import cross_validate  # evaluation tool

# Load Anime dataset files (must be in same folder as notebook)
ratings = pd.read_csv("rating.csv")      # user-anime ratings
anime = pd.read_csv("anime.csv")        # anime metadata (titles, etc.)

# Preview dataset structure
print(ratings.head())
print(anime.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.0 MB/s eta 0:00:00
   user_id  anime_id  rating
0        1        20      -1
1        1        24      -1
2        1        79      -1
3        1       226      -1
4        1       241      -1
   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...  

## **PART 2: DATA CLEANING**

In [2]:
# Remove missing/invalid ratings (-1 means "watched but not rated")
ratings = ratings[ratings["rating"] != -1]

# Sample data to speed up training since it might take excessive time to run and load
ratings = ratings.sample(n=100000, random_state=42)

# Check dataset size after cleaning
print("Cleaned ratings size:", len(ratings))

Cleaned ratings size: 100000


## **PART 3: DEVELOP SURPRISE DATASET**

In [3]:
# Define rating scale (Anime dataset uses 1–10 ratings)
reader = Reader(rating_scale=(1, 10))

# Convert dataframe into Surprise format
data = Dataset.load_from_df(
    ratings[["user_id", "anime_id", "rating"]],
    reader
)

## **PART 4: BASELINE MODELS (DELIVERABLE 1)**

In [4]:
# -------------------------
# SVD MODEL
# -------------------------
svd = SVD()

svd_results = cross_validate(
    svd,
    data,
    measures=["RMSE", "MAE"],
    cv=5,
    verbose=True
)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.3390  1.3378  1.3322  1.3428  1.3285  1.3361  0.0051  
MAE (testset)     1.0311  1.0281  1.0283  1.0296  1.0201  1.0274  0.0038  
Fit time          1.32    1.23    1.22    1.24    1.22    1.25    0.04    
Test time         0.21    0.20    0.11    0.11    0.19    0.16    0.05    


In [5]:
import pandas as pd # Added to ensure pandas is available for data loading
from surprise import Dataset # Added to ensure Dataset is available for 'data' creation
from surprise import Reader # Added to ensure Reader is available for 'reader' creation
from surprise import KNNBasic # Added to resolve NameError
from surprise.model_selection import cross_validate # Added to resolve NameError

# Re-load and clean ratings if 'ratings' is not defined (due to kernel restart)
try:
    _ = ratings # Attempt to access 'ratings' to check if it's defined
except NameError:
    print("Warning: 'ratings' not found, re-loading and cleaning data...")
    ratings = pd.read_csv("rating.csv")
    ratings = ratings[ratings["rating"] != -1]
    ratings = ratings.sample(n=100000, random_state=42)
    print("Cleaned ratings size:", len(ratings))

# Re-define reader if 'reader' is not defined
try:
    _ = reader # Attempt to access 'reader'
except NameError:
    print("Warning: 'reader' not found, re-defining Reader...")
    reader = Reader(rating_scale=(1, 10))

# Re-define data if 'data' is not defined
try:
    _ = data # Attempt to access 'data'
except NameError:
    print("Warning: 'data' not found, re-creating Surprise dataset...")
    data = Dataset.load_from_df(
        ratings[["user_id", "anime_id", "rating"]],
        reader
    )

# -------------------------
# KNN MODEL
# -------------------------
knn = KNNBasic()

knn_results = cross_validate(
    knn,
    data,
    measures=["RMSE", "MAE"],
    cv=5,
    verbose=True
)


Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Evaluating RMSE, MAE of algorithm KNNBasic on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.6517  1.6675  1.6734  1.6641  1.6690  1.6651  0.0074  
MAE (testset)     1.2674  1.2722  1.2778  1.2720  1.2753  1.2729  0.0035  
Fit time          11.22   7.69    7.72    7.60    6.38    8.12    1.63    
Test time         3.17    1.72    1.59    2.36    1.35    2.04    0.66    


In [6]:
# Compute average metrics
svd_rmse = np.mean(svd_results["test_rmse"])
svd_mae = np.mean(svd_results["test_mae"])

knn_rmse = np.mean(knn_results["test_rmse"])
knn_mae = np.mean(knn_results["test_mae"])

print("\nBASELINE RESULTS")
print("SVD RMSE:", svd_rmse)
print("SVD MAE:", svd_mae)

print("KNN RMSE:", knn_rmse)
print("KNN MAE:", knn_mae)


BASELINE RESULTS
SVD RMSE: 1.3360519415062708
SVD MAE: 1.0274390241480016
KNN RMSE: 1.6651419543000852
KNN MAE: 1.2729486218123822


## **PART 5: NOVELTY FILTER (DELIVERABLE 2)**

In [7]:
# Count how many ratings each anime has (popularity measure)
anime_popularity = ratings["anime_id"].value_counts()

# Get top 20 most popular anime IDs
top_20_anime = anime_popularity.head(20).index

# Remove popular anime to increase novelty
ratings_novel = ratings[~ratings["anime_id"].isin(top_20_anime)]

# Show dataset reduction
print("Original ratings:", len(ratings))
print("After novelty filter:", len(ratings_novel))

Original ratings: 100000
After novelty filter: 93065


## **PART 6: REBUILD DATASET + RETRAIN**

In [8]:
# Convert filtered dataset into Surprise format
novel_data = Dataset.load_from_df(
    ratings_novel[["user_id", "anime_id", "rating"]],
    reader
)

# Train SVD again on novelty dataset
novel_svd = SVD()

novel_results = cross_validate(
    novel_svd,
    novel_data,
    measures=["RMSE", "MAE"],
    cv=5,
    verbose=True
)

# Extract metrics
novel_rmse = np.mean(novel_results["test_rmse"])
novel_mae = np.mean(novel_results["test_mae"])

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.3503  1.3257  1.3345  1.3307  1.3558  1.3394  0.0116  
MAE (testset)     1.0384  1.0204  1.0228  1.0291  1.0403  1.0302  0.0080  
Fit time          1.90    3.76    1.75    1.18    1.18    1.95    0.95    
Test time         0.40    0.46    0.11    0.10    0.25    0.26    0.15    


## **PART 7: FINAL COMPARISON (DELIVERABLE 3)**

In [9]:
# Create comparison table
results = pd.DataFrame({
    "Model": ["SVD", "KNNBasic", "SVD + Novelty"],
    "RMSE": [svd_rmse, knn_rmse, novel_rmse],
    "MAE": [svd_mae, knn_mae, novel_mae]
})

print("\nFINAL COMPARISON")
print(results)


FINAL COMPARISON
           Model      RMSE       MAE
0            SVD  1.336052  1.027439
1       KNNBasic  1.665142  1.272949
2  SVD + Novelty  1.339402  1.030201


## **PART 8: SUMMARY LOGIC**

In [10]:
# Simple interpretation of results

if svd_rmse < knn_rmse:
    print("\nSVD performs better than KNNBasic in accuracy.")
else:
    print("\nKNNBasic performs better than SVD in accuracy.")

print("\nNovelty Effect:")
print("Removing popular anime increases diversity but may slightly reduce accuracy.")


SVD performs better than KNNBasic in accuracy.

Novelty Effect:
Removing popular anime increases diversity but may slightly reduce accuracy.


## **PART 9: DELIVERABLE 4**

Throughout this project, 2 methods were utilized to recommend anime shows using SVD and KNNBasic. I used a list of anime shows to see how well these methods worked. I checked how well SVD and KNNBasic did using RMSE and MAE.

SVD did a job than KNNBasic.

* I wanted to make anime recommendations more interesting for people who like watching various animes.

So I took out 20 anime shows from the list. This helped the system focus on these anime shows. The suggestions became more varied and interesting. Making predictions a bit accurate is okay. It helps people find anime shows they like. This is really important for tech real-world systems. When we test these systems on a computer we use RMSE and MAE. These metrics show how accurate SVD and KNNBasic are. They do not show if people are satisfied with the anime shows they find.

To test SVD and KNNBasic with people we could look at:

* How often people click on anime shows

* How often people watch anime shows

* How often people look at anime shows

* How long people stay on the site

* If people come back to the site


We could test SVD and KNNBasic by splitting people into two groups.

One group uses SVD to find anime shows. The other group uses KNNBasic.

Then, I compared how engaged people are with SVD and KNNBasic.

This helps us see which anime show suggestion system is better.

SVD and KNNBasic are anime show suggestion methods we are testing.

We want to see which one works best. The goal is to make people elated with the anime shows they find. The main purpose is to compare SVD and KNNBasic to see which one is better, for suggesting anime shows.